In [ ]:
import gc
import os
from pathlib import Path

import torch
from PIL import Image
from IPython.display import display
from diffusers import DiffusionPipeline
from diffusers.quantizers import PipelineQuantizationConfig
import random

## 1. Конфиги
`MODEL_MODE`: `"quantized"` (NF4) либо `"normal"` (BF16). 

Предобработка изображнения: 
Масштабирование до минимальной стороны в 256 пикселя и кратности 16 `PREPROCESS_MODE`: `"resize"`
Добавление черных полей до минималки 64 пикселя по стороне `PREPROCESS_MODE`: `"pad"`.

С маленькими картинками(до 100х100 точно) плохо модель работает, для более адекватных генераций рекомендую использовать масштабирование

RANDOM_SEED обеспечивает случайное зерно при генерации, одинаковые зёрна при генереации будут давать одинаковый результат

NUM_INFERENCE_STEPS = 10 - количество шагов численного решения процесса генерации изображения. Лучше ставить больше 15 - для адекватных замен текста, по моим тестам после 30 большой разницы уже нет

NUM_ATTEMPTS = 3 - кол-во доп генераций с другими сидами, для более честного эксперимента

In [ ]:
MODEL_ID = "black-forest-labs/FLUX.2-klein-base-4B"
MODEL_MODE = "quantized"   # "quantized" | "normal"
PREPROCESS_MODE = "resize"    # "pad" | "resize"
RANDOM_SEED = True 

ROOT_DIR = Path(r"E:\Workowoe\SHIFT\Research_\flux2")
INPUT_DIR = ROOT_DIR / "input"
OUTPUT_DIR = ROOT_DIR / "output"
IMAGES_LIST_PATH = ROOT_DIR / "images.txt"
PROMPT_PATH = ROOT_DIR / "prompt.txt"

REPLACE_PROMPT_TEMPLATE = (
    'Replace the text "OLD" with "NEW". '
    'Keep the same FONT, size, weight, color, letter spacing, alignment, '
    'background texture, scan noise and perspective. Change only the necessary text area.'
)
CLEAN_BACKGROUND_PROMPT_TEMPLATE = (
    'Remove the text "{src}" completely. Do not add any new text. '
    'Reconstruct and preserve the original background naturally. '
    'Keep the same color, paper texture, scan noise, compression artifacts, '
    'lighting, alignment and perspective. Change only the necessary text area.'
)
EMPTY_MARKERS = {"", "[SPACE]", "[EMPTY]", "[BLANK]", "<SPACE>", "<EMPTY>"}

MIN_SIDE_RESIZE = 256
MAX_SIDE_RESIZE = 1024

MIN_SIDE_PAD = 80
CROP_PADDED_RESULT = True  

NUM_INFERENCE_STEPS = 10
GUIDANCE_SCALE = 4.0
NUM_ATTEMPTS = 3
BASE_SEED = 0
DTYPE = torch.bfloat16
SAVE_INDIVIDUAL_OUTPUTS = True
SHOW_COLLAGES = True
COLLAGE_PADDING = 20
OUTPUT_SUFFIX = "_edited"
COLLAGE_SUFFIX = "_collage"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Загрузка модели
При повторном запуске ячейки с тем же режимом и ID уже существующий `pipe` используется повторно. Переключение BF16/NF4 приведёт к повторной загрузке; при недостатке памяти безопаснее **Restart Kernel**, затем запуск с новыми настройками. Т.к. квантизованная модель из-за какой-то ошибки в любом случае будет иметь утечку

In [ ]:
loaded_signature = globals().get("_loaded_signature")
signature = (MODEL_ID, MODEL_MODE, str(DTYPE))
if "pipe" in globals() and loaded_signature == signature:
    print("Модель уже загружена; повторно не загружаю.")
else:
    if "pipe" in globals():
        del pipe
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    kwargs = {"dtype": DTYPE}
    if MODEL_MODE == "quantized":
        kwargs["quantization_config"] = PipelineQuantizationConfig(
            quant_backend="bitsandbytes_4bit",
            quant_kwargs={
                "load_in_4bit": True,
                "bnb_4bit_quant_type": "nf4",
                "bnb_4bit_compute_dtype": DTYPE,
            },
            components_to_quantize=["transformer", "text_encoder"],
        )
    elif MODEL_MODE != "normal":
        raise ValueError("MODEL_MODE должен быть 'normal' или 'quantized'")
    print(f"Загрузка {MODEL_ID} / {MODEL_MODE}...")
    pipe = DiffusionPipeline.from_pretrained(MODEL_ID, **kwargs)
    pipe.enable_model_cpu_offload()
    if getattr(pipe, "vae", None) is not None:
        pipe.vae.enable_tiling()
    _loaded_signature = signature
    print("Модель готова")


## 4. Функции: подготовка, шаблоны и коллаж

In [ ]:
def read_jobs(path):
    jobs = []
    for line_num, raw in enumerate(Path(path).read_text(encoding="utf-8-sig").splitlines(), 1):
        if not raw.strip() or raw.lstrip().startswith("#"):
            continue
        parts = raw.split("|", maxsplit=2)
        if len(parts) != 3:
            raise ValueError(f"Строка {line_num}: ожидается filename|A|B, получено {raw!r}")
        name, src, dst = (part.strip() for part in parts)
        if not name or not src:
            raise ValueError(f"Строка {line_num}: имя файла и A не должны быть пустыми")
        jobs.append((name, src, dst))
    return jobs


def get_template():
    # В исходном script.py шаблон замены читался из prompt.txt, а не находился в коде.
    if PROMPT_PATH.is_file():
        template = PROMPT_PATH.read_text(encoding="utf-8-sig").strip()
        if not template:
            raise ValueError("prompt.txt пуст")
        return template
    return REPLACE_PROMPT_TEMPLATE


def make_prompt(template, src, dst):
    if dst.upper() in EMPTY_MARKERS:
        return CLEAN_BACKGROUND_PROMPT_TEMPLATE.format(src=src)
    # Подстановка одновременно, чтобы A не перезаписывалась второй заменой
    return (template.replace("OLD", "{__SOURCE__}")
                    .replace("NEW", "{__TARGET__}")
                    .replace("{__SOURCE__}", src)
                    .replace("{__TARGET__}", dst))


def resize_for_flux(img, min_side=256, max_side=1024):
    w, h = img.size
    if min(w, h) < min_side:
        s = min_side / min(w, h)
        w, h = round(w * s), round(h * s)
    if max(w, h) > max_side:
        s = max_side / max(w, h)
        w, h = round(w * s), round(h * s)
    w, h = max(64, round(w / 16) * 16), max(64, round(h / 16) * 16)
    return img.resize((w, h), Image.Resampling.LANCZOS) if (w, h) != img.size else img.copy()


def pad_for_flux(img, min_side=80):
    """Без масштабирования: чёрная подложка, исходник в центре."""
    w, h = img.size
    w2 = max(min_side, ((w + 15) // 16) * 16)
    h2 = max(min_side, ((h + 15) // 16) * 16)
    x, y = (w2 - w) // 2, (h2 - h) // 2
    canvas = Image.new("RGB", (w2, h2), (0, 0, 0))
    canvas.paste(img, (x, y))
    return canvas, (x, y, x + w, y + h)


def prepare_image(img):
    if PREPROCESS_MODE == "resize":
        result = resize_for_flux(img, MIN_SIDE_RESIZE, MAX_SIDE_RESIZE)
        return result, None
    if PREPROCESS_MODE == "pad":
        return pad_for_flux(img, MIN_SIDE_PAD)
    raise ValueError("PREPROCESS_MODE должен быть 'resize' или 'pad'")


def build_collage(images, padding=20):
    widths = [im.width for im in images]
    heights = [im.height for im in images]
    canvas = Image.new("RGB", (sum(widths) + (len(images) + 1) * padding,
                               max(heights) + 2 * padding), (255, 255, 255))
    x = padding
    for im in images:
        canvas.paste(im, (x, padding + (max(heights) - im.height) // 2))
        x += im.width + padding
    return canvas


def output_base(image_path):
    try:
        relative = image_path.resolve().relative_to(INPUT_DIR.resolve())
    except ValueError:
        relative = Path(image_path.name)
    base = OUTPUT_DIR / relative.parent / relative.stem
    base.parent.mkdir(parents=True, exist_ok=True)
    return base


## 5. Просмотр списка задач и фактических промптов
Для обычной замены используется `prompt.txt`, если он есть; иначе `REPLACE_PROMPT_TEMPLATE` из настроек. Для удаления — отдельный `CLEAN_BACKGROUND_PROMPT_TEMPLATE`.

In [ ]:
REPLACE_PROMPT_TEMPLATE = (
    'Replace the text "OLD" with "NEW". '
    'Keep the same FONT, size, weight, color, letter spacing, alignment, '
    'background texture, scan noise and perspective. Change only the necessary text area.'
)
CLEAN_BACKGROUND_PROMPT_TEMPLATE = (
    'Remove the text "{src}" completely. Do not add any new text. '
    'Reconstruct and preserve the original background naturally. '
    'Keep the same color, paper texture, scan noise, compression artifacts, '
    'lighting, alignment and perspective. Change only the necessary text area.'
)
PREPROCESS_MODE = "resize"    # "pad" | "resize"
RANDOM_SEED = True 

In [ ]:
jobs = read_jobs(IMAGES_LIST_PATH)
template = get_template()
print(f"Задач: {len(jobs)}; источник промпта: {'prompt.txt' if PROMPT_PATH.is_file() else 'REPLACE_PROMPT_TEMPLATE'}")
for image_name, src, dst in jobs:
    print(f"\n{image_name} | {src!r} -> {dst!r}")
    print(make_prompt(template, src, dst))


## 6. Запуск обработки одного изображения

In [ ]:
def run_job(job, show=True):
    image_name, src, dst = job
    image_path = Path(image_name)
    if not image_path.is_absolute():
        image_path = INPUT_DIR / image_path
    if not image_path.is_file():
        raise FileNotFoundError(image_path)
    with Image.open(image_path) as opened:
        original = opened.convert("RGB")
    prepared, crop_box = prepare_image(original)
    prompt = make_prompt(get_template(), src, dst)
    print(f"{image_path.name}: {original.size} -> {prepared.size}; режим {PREPROCESS_MODE}")
    print("Промпт:", prompt)

    # Изображение слева соответствует изображению, которое подали модели.
    reference = prepared
    results = []
    base = output_base(image_path)
    for i in range(NUM_ATTEMPTS):
        if RANDOM_SEED == True:
            seed = random.randint(1,10**6)
        else:
            seed = BASE_SEED + i
        print(f"Попытка {i+1}/{NUM_ATTEMPTS}, seed={seed}")
        generator = torch.Generator(device="cuda").manual_seed(seed)
        with torch.inference_mode():
            generated = pipe(image=prepared, prompt=prompt,
                             guidance_scale=GUIDANCE_SCALE,
                             num_inference_steps=NUM_INFERENCE_STEPS,
                             generator=generator).images[0].convert("RGB")
        if crop_box is not None and CROP_PADDED_RESULT:
            # На случай несовпадения размеров выходного изображения с входом.
            if generated.size != prepared.size:
                generated = generated.resize(prepared.size, Image.Resampling.LANCZOS)
            generated = generated.crop(crop_box)
            reference = original
        results.append(generated)
        if SAVE_INDIVIDUAL_OUTPUTS:
            generated.save(base.with_name(f"{base.name}{OUTPUT_SUFFIX}_{PREPROCESS_MODE}_{MODEL_MODE}_{NUM_INFERENCE_STEPS}_try{i+1}.png"))

    collage = build_collage([reference, *results], COLLAGE_PADDING)
    collage_path = base.with_name(f"{base.name}{COLLAGE_SUFFIX}_{PREPROCESS_MODE}_{MODEL_MODE}_{NUM_INFERENCE_STEPS}.png")
    collage.save(collage_path)
    print("Коллаж:", collage_path)
    if show:
        display(collage)
    return collage_path

JOB_INDEX = 0
run_job(jobs[JOB_INDEX])


## 7. Пакетная обработка всех изображений

In [ ]:
# Раскомментируй для пакетного запуска.
# for idx, job in enumerate(jobs, 1):
#     print(f"\n=== {idx}/{len(jobs)} ===")
#     try:
#         run_job(job, show=SHOW_COLLAGES)
#     except Exception as exc:
#         print(f"[ERROR] {job[0]}: {type(exc).__name__}: {exc}")
#     gc.collect()
